# Lesson 2: Multiple Nodes & TypedDict State Schema

## Objective
Extend the graph to support three arithmetic operations — **add, multiply, divide** — as separate nodes,
all running sequentially in one graph, demonstrating partial state updates and richer schemas.

## Limitation of Lesson 1
- ❌ Only one hardcoded operation (add)
- ❌ Single-field result — no room for multiple outputs
- ❌ No sequential node chaining

## What is NEW in Lesson 2?
- ✅ **Multiple nodes** in a single graph
- ✅ **Sequential chaining**: `node_A → node_B → node_C`
- ✅ **Richer TypedDict** with `Optional` fields
- ✅ Deep understanding of **partial state updates** (each node only touches its own fields)
- ❌ Still no conditional routing — all three run every time (fixed in Lesson 3)

## Theory: Immutable State Merging

When `add_node` returns `{"sum_result": 8}`, LangGraph does NOT replace the entire state.  
It **merges** the returned dict into the current state:

```
Before add_node:   {a:5, b:3, sum_result:None, product_result:None, division_result:None}
add_node returns:  {"sum_result": 8}
After merge:       {a:5, b:3, sum_result:8,    product_result:None, division_result:None}
                                                ↑                    ↑
                                           unchanged              unchanged
```

Each node is **isolated** — it only reads what it needs and writes what it produces.  
This makes nodes independently testable, replaceable, and parallelizable.


In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional
from IPython.display import Image, display

print("✓ Imports successful")
print("  Optional[int] = field may be None (not yet computed)")


In [ ]:
# ── Cell 2: Richer State Schema ─────────────────────────────────────────────
# Optional[T] means the field can be T or None
# We use None as the "not yet computed" sentinel
# Each node is responsible for filling ONE of these fields

class State(TypedDict):
    a: int                            # First operand (input)
    b: int                            # Second operand (input)
    operation: str                    # Which op was requested (informational)
    sum_result: Optional[int]         # Result of addition
    product_result: Optional[int]     # Result of multiplication
    division_result: Optional[float]  # Result of division

print("✓ State schema defined")
print("Fields:", list(State.__annotations__.keys()))


In [ ]:
# ── Cell 3: Three Independent Arithmetic Nodes ──────────────────────────────
# Each node:
#   - Reads only the fields it needs (a, b)
#   - Writes only the field it produces (xxx_result)
#   - Is completely independent of the other nodes

def add_node(state: State) -> dict:
    result = state["a"] + state["b"]
    print(f"  [add]      {state['a']} + {state['b']} = {result}")
    return {"sum_result": result}          # Partial update: only sum_result

def multiply_node(state: State) -> dict:
    result = state["a"] * state["b"]
    print(f"  [multiply] {state['a']} × {state['b']} = {result}")
    return {"product_result": result}      # Partial update: only product_result

def divide_node(state: State) -> dict:
    if state["b"] == 0:
        print(f"  [divide]   ERROR: division by zero")
        return {"division_result": None}   # Safe — returns None
    result = state["a"] / state["b"]
    print(f"  [divide]   {state['a']} ÷ {state['b']} = {result:.4f}")
    return {"division_result": result}     # Partial update: only division_result

print("✓ Three nodes defined")
print("  Each node is independent — no node depends on another node's output")


## Build the Sequential Graph

Graph topology:
```
START → add → multiply → divide → END
```

All three operations run in sequence on every invocation.
Lesson 3 adds routing so only ONE operation runs per call.


In [ ]:
# ── Cell 4: Build Sequential Graph ─────────────────────────────────────────
builder = StateGraph(State)

# Register all three nodes
builder.add_node("add",      add_node)
builder.add_node("multiply", multiply_node)
builder.add_node("divide",   divide_node)

# Wire sequential chain: START → add → multiply → divide → END
builder.add_edge(START,      "add")
builder.add_edge("add",      "multiply")
builder.add_edge("multiply", "divide")
builder.add_edge("divide",    END)

graph = builder.compile()
print("✓ Sequential 3-node pipeline compiled")
print(f"  Nodes: {list(graph.get_graph().nodes.keys())}")


In [ ]:
# ── Cell 5: Visualize ───────────────────────────────────────────────────────
# You should see a straight line: __start__ → add → multiply → divide → __end__
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 6: Run the Graph ───────────────────────────────────────────────────
initial_state = {
    "a": 12,
    "b": 4,
    "operation": "all",          # All three run — no routing yet
    "sum_result": None,
    "product_result": None,
    "division_result": None,
}

print("Running sequential graph with a=12, b=4")
print("-" * 45)

final_state = graph.invoke(initial_state)

print("-" * 45)
print(f"\n  a = {final_state['a']}, b = {final_state['b']}")
print(f"  12 + 4  = {final_state['sum_result']}")
print(f"  12 × 4  = {final_state['product_result']}")
print(f"  12 ÷ 4  = {final_state['division_result']:.4f}")


In [ ]:
# ── Cell 7: Verify Partial Update Isolation ─────────────────────────────────
# Demonstrate that each node ONLY updated its own field
# Check: after add runs, multiply and divide fields are still None
# (This shows partial updates — nodes don't clobber each other)

print("Verifying partial update isolation...")
print()
print("After all 3 nodes run:")
print(f"  sum_result      = {final_state['sum_result']}    ← set by add_node")
print(f"  product_result  = {final_state['product_result']}    ← set by multiply_node")
print(f"  division_result = {final_state['division_result']} ← set by divide_node")
print()
print("Each node wrote exactly ONE field — none interfered with others ✓")


In [ ]:
# ── Cell 8: Test Divide by Zero Safety ──────────────────────────────────────
state_zero = {
    "a": 10, "b": 0, "operation": "all",
    "sum_result": None, "product_result": None, "division_result": None
}

print("Testing b=0 (division by zero case):")
out = graph.invoke(state_zero)
print(f"  sum_result      = {out['sum_result']}")
print(f"  product_result  = {out['product_result']}")
print(f"  division_result = {out['division_result']}  ← safely returned None")


In [ ]:
# ── Cell 9: Multiple Test Cases ─────────────────────────────────────────────
print(f"{'a':>6} {'b':>6} {'a+b':>8} {'a×b':>8} {'a÷b':>10}")
print("-" * 45)

for a, b in [(5, 2), (10, 3), (7, 7), (100, 25)]:
    state = {"a":a,"b":b,"operation":"all","sum_result":None,"product_result":None,"division_result":None}
    out = graph.invoke(state)
    div_str = f"{out['division_result']:.4f}" if out['division_result'] is not None else "N/A"
    print(f"{a:>6} {b:>6} {out['sum_result']:>8} {out['product_result']:>8} {div_str:>10}")


## Code Explanation

### The Partial Update Pattern
```python
def add_node(state: State) -> dict:
    return {"sum_result": state["a"] + state["b"]}
    #       ↑ Only sum_result is in the return dict
    #         product_result and division_result are NOT touched
```

LangGraph merge logic:
```python
new_state = {**current_state, **node_return_value}
# {a:12, b:4, sum_result:None, product_result:None, division_result:None}
# merged with {"sum_result": 16}
# = {a:12, b:4, sum_result:16, product_result:None, division_result:None}
```

### Why Optional[int]?
`Optional[int]` = `Union[int, None]`  
Using `None` (not `0`) as the "not yet computed" sentinel prevents false results.  
`0` could be a valid answer; `None` unambiguously means "not computed yet".

## Summary

| | Lesson 1 | Lesson 2 |
|--|---------|----------|
| Nodes | 1 | 3 |
| Operations | Add only | Add + Multiply + Divide |
| Graph shape | Linear (1 node) | Linear (3 nodes in sequence) |
| State fields | 3 | 6 (with Optional) |

## Limitations → What Lesson 3 Solves
- ❌ All three nodes always run — wasteful
- ❌ No way to select one operation based on user input
- ❌ **Lesson 3**: `add_conditional_edges` routes execution to exactly ONE node
